# 03 DESeq2 で2つの条件を比べる

shControl と shPPP3CA で、遺伝子ごとに RNA 量を比べます。RNA-seq のカウントの比較によく使われる DESeq2 という方法を、Python 版の PyDESeq2 で実行します。

DESeq2 は、次の3つを順に行います。

1. 試料ごとの読み取った量の違いを調整する係数（サイズファクター）を求める
2. 遺伝子ごとに、同じ条件の試料のあいだのばらつきを見積もる
3. 条件の違いによる変化の大きさ（log₂ fold change）と、その確からしさ（p値）を求める

## 準備

下のセルを実行して、配布元から `samples.csv` をダウンロードします。

In [ ]:
%pip install -q pydeseq2==0.5.4 anndata==0.12.10
import warnings
warnings.filterwarnings("ignore")  # 計算には影響しない警告を表示しない

DATA_URL = "https://pub-fcc6b90fb32448a4b3b455e664df76f0.r2.dev/e2f1-rnaseq/v1/"
!wget -q -nc {DATA_URL}samples.csv
!ls -l samples.csv

## 前のノートブックの結果を読み込む

02_QC_PCA の最後でダウンロードした `results_02.zip` を使います。下のセルを実行し、「ファイルを選択」を押して `results_02.zip` を選んでください。zip の中の `counts_filtered.csv`, `vst.csv` を、このノートブックで使います。

In [ ]:
import zipfile
import os
from pathlib import Path

PREV_ZIP = "results_02.zip"
NEED = ['counts_filtered.csv', 'vst.csv']

if not all(Path(n).exists() for n in NEED):
    if Path(PREV_ZIP).exists():
        zipfile.ZipFile(PREV_ZIP).extractall()
    else:
        from google.colab import files
        for name in files.upload():
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall()

missing = [n for n in NEED if not Path(n).exists()]
if missing:
    print("見つからないファイルがあります:", missing)
    print("このセルをもう一度実行して zip を選び直すか、下の「zip が手元にないとき」のセルを実行してください。")
else:
    print("読み込みました:", NEED)

### zip が手元にないとき

前のノートブックを最後まで実行できなかった人や、zip をなくした人だけ、下のセルを実行してください。教員が用意した `results_02.zip` をダウンロードして使います。

In [ ]:
!wget -q -nc {DATA_URL}results_02.zip
zipfile.ZipFile(PREV_ZIP).extractall()
print("読み込みました:", [n for n in NEED if Path(n).exists()])

In [ ]:
import pandas as pd

counts = pd.read_csv("counts_filtered.csv", index_col="gene")
samples = pd.read_csv("samples.csv", index_col="sample")
samples = samples.loc[counts.columns]
print(counts.shape[0], "遺伝子 ×", counts.shape[1], "試料")
samples

## DESeq2 を実行する

`design="~condition"` は、「`condition` 列の違いで比べる」という指定です。カウント表は、PyDESeq2 の決まりに合わせて、行が試料・列が遺伝子になるように向きを変えて（`.T`）渡します。

実行には1分ほどかかります。

In [ ]:
from pydeseq2.dds import DeseqDataSet

dds = DeseqDataSet(counts=counts.T, metadata=samples, design="~condition", quiet=True)
dds.deseq2()
print("終わりました")

## サイズファクター

サイズファクターは、試料ごとの読み取った量の違いを表す係数です。カウントをサイズファクターで割ったものを、正規化したカウントと呼びます。

DESeq2 のサイズファクターは、合計で割るだけの計算ではありません。遺伝子ごとに「この試料の値 ÷ 全試料の平均的な値」の比を求め、その中央値を使います。こうすると、ごく一部の遺伝子のカウントが極端に多くても、その影響を受けにくくなります。

In [ ]:
size_factors = pd.DataFrame({
    "size_factor": dds.obs["size_factors"],
    "total_counts": counts.sum(),
})
size_factors["total / mean(total)"] = size_factors["total_counts"] / size_factors["total_counts"].mean()
size_factors.round(3)

サイズファクターと、合計を平均で割った値を見比べてください。

## 正規化したカウント

*PPP3CA* について、元のカウントと、正規化したカウントを並べます。

In [ ]:
normalized = pd.DataFrame(dds.layers["normed_counts"].T, index=counts.index, columns=counts.columns)

GENE = "PPP3CA"
pd.DataFrame({"raw": counts.loc[GENE], "normalized": normalized.loc[GENE].round(1)})

## shPPP3CA と shControl を比べる

`contrast=["condition", "shPPP3CA", "shControl"]` で、「shPPP3CA ÷ shControl」の向きで比べます。log₂ fold change が正なら shPPP3CA で増え、負なら減っています。

In [ ]:
from pydeseq2.ds import DeseqStats

stats = DeseqStats(dds, contrast=["condition", "shPPP3CA", "shControl"], quiet=True)
stats.summary()
res = stats.results_df.sort_values("padj")
res.head(10)

結果の表の列の意味は次のとおりです。

| 列 | 意味 |
|---|---|
| `baseMean` | 6試料の正規化したカウントの平均 |
| `log2FoldChange` | 変化の大きさ。2倍なら +1、同じなら 0、半分なら −1 |
| `lfcSE` | log2FoldChange の見積もりの不確かさ（標準誤差） |
| `stat` | log2FoldChange ÷ lfcSE。検定に使う値 |
| `pvalue` | 本当は差がないとしたときに、これくらい以上の差が偶然に出る確率 |
| `padj` | たくさんの遺伝子を同時に調べたことを考えて調整した p値（Benjamini–Hochberg 法） |

`padj` が `NaN`（空欄）の遺伝子は、外れ値があるなどの理由で、DESeq2 が検定から外したものです。

## 変化した遺伝子を数える

ここでは「padj < 0.05」を、変化があったとみなす基準にします。さらに「変化の大きさが2倍以上（|log₂FC| ≥ 1）」も加えた数も数えます。どちらもこの実習で決めた基準です。

In [ ]:
PADJ = 0.05
LFC = 1

sig = res[res["padj"] < PADJ]
print(f"padj < {PADJ}:", len(sig), "遺伝子")
print("  そのうち shPPP3CA で増加:", (sig["log2FoldChange"] > 0).sum())
print("  そのうち shPPP3CA で減少:", (sig["log2FoldChange"] < 0).sum())
print(f"padj < {PADJ} かつ |log2FC| >= {LFC}:", (sig["log2FoldChange"].abs() >= LFC).sum(), "遺伝子")

## 遺伝子を調べる

`GENES` に遺伝子名を加えると、その遺伝子の結果を並べて見られます。

In [ ]:
GENES = ["PPP3CA", "E2F1", "GAPDH"]

res.loc[GENES]

## 確かめよう

1. *PPP3CA* の log2FoldChange はいくつですか。何分の1に減ったことになりますか（ヒント：2 の −log2FoldChange 乗）。
2. *E2F1* の RNA 量は、どれくらい変わっていますか。論文の「カルシニューリンが E2F1 のタンパク質を安定にする」という話と合わせて考えてみましょう。RNA-seq で測っているのは何でしたか。
3. `PADJ` を 0.01 に変えると、変化した遺伝子の数はどう変わりますか。

In [ ]:
res.to_csv("deseq2_results.csv")
normalized.to_csv("normalized_counts.csv")

## 結果をダウンロードする

下のセルを実行すると、ここまでの結果をまとめた `results_03.zip` がパソコンにダウンロードされます。次の 04_Visualization の最初でアップロードするので、保存した場所を覚えておいてください。

In [ ]:
import zipfile

OUT = "results_03.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ['counts_filtered.csv', 'vst.csv', 'deseq2_results.csv', 'normalized_counts.csv']:
        z.write(name)
print(OUT, "に入れたファイル:", zipfile.ZipFile(OUT).namelist())

try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass  # Colab 以外で実行したときは、手元のフォルダに保存される

次は **04_Visualization** に進みます。